In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Master/LTAINC/lightweight-medical-model')
OUTPUTS_ROOT = PROJECT_DIR / 'outputs'
DATASET_DIR = PROJECT_DIR / 'data/busi'
EVALUATION_ROOT = PROJECT_DIR / 'medical-evaluation-results'

assert (PROJECT_DIR / 'evaluate_medical_metrics.py').is_file(), 'Thiếu evaluator trong code mới'
assert OUTPUTS_ROOT.is_dir(), f'Không tìm thấy outputs: {OUTPUTS_ROOT}'
assert DATASET_DIR.is_dir(), f'Không tìm thấy BUSI: {DATASET_DIR}'
print('Project:', PROJECT_DIR)
print('Outputs:', OUTPUTS_ROOT)
print('Dataset:', DATASET_DIR)
print('Evaluation:', EVALUATION_ROOT)

In [ ]:
%cd {PROJECT_DIR}
# Bỏ comment nếu cần cập nhật code trước khi chạy.
# !git pull origin main

# Detailed evaluation / medical metrics

Chạy toàn bộ checkpoint có classification head. Raw là kết quả chính; refined là ablation cho model multi-task.

In [ ]:
!python run_all_medical_evaluations.py \
  --outputs-root "$OUTPUTS_ROOT" \
  --dataset-dir "$DATASET_DIR" \
  --output-root "$EVALUATION_ROOT" \
  --batch-size 8 \
  --num-workers 2 \
  --threshold 0.5 \
  --area-threshold 0.005

## Kiểm tra trạng thái và bảng tổng hợp

In [ ]:
import pandas as pd
from IPython.display import display

status = pd.read_csv(EVALUATION_ROOT / 'evaluation_status.csv')
display(status[['config', 'model', 'status', 'return_code']])

failed = status[status.status != 'completed']
if not failed.empty:
    print('Các cấu hình lỗi:')
    display(failed[['config', 'checkpoint', 'error']])
else:
    print('Tất cả cấu hình đã hoàn thành.')

In [ ]:
summary = pd.read_csv(EVALUATION_ROOT / 'all_summary.csv')
per_class = pd.read_csv(EVALUATION_ROOT / 'all_per_class_metrics.csv')

print('RAW summary')
display(summary[summary.variant == 'raw'][[
    'config', 'accuracy', 'macro_f1', 'macro_auc_ovr',
    'dice_lesion', 'iou_lesion'
]])

print('Malignant medical metrics — RAW')
display(per_class[(per_class.variant == 'raw') & (per_class['class'] == 'malignant')][[
    'config', 'precision', 'sensitivity', 'specificity', 'f1', 'auc', 'support'
]])

## Xem confusion matrix raw

In [ ]:
from IPython.display import Image, display

for matrix_path in sorted(EVALUATION_ROOT.glob('*/confusion_matrix_raw.png')):
    print(matrix_path.parent.name)
    display(Image(filename=str(matrix_path)))